<a href="https://colab.research.google.com/github/TensorCruncher/animal-image-search/blob/main/image_search_caption.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [ ]:
!pip install faiss-cpu open-clip-torch torchvision -q

In [ ]:
!wget https://raw.githubusercontent.com/TensorCruncher/animal-image-search/refs/heads/main/image_paths.json
!wget https://raw.githubusercontent.com/TensorCruncher/animal-image-search/refs/heads/main/image_embeddings.npy
!wget -O dog.jpg https://i.pinimg.com/736x/64/9e/1b/649e1b9a4ff0de136eba812c527eb9e8.jpg

In [ ]:
import torch
import open_clip
import faiss

import numpy as np
import json

from IPython.display import display, Image as IPyImage
from PIL import Image as PILImage
import io

from google.colab import drive

In [ ]:
drive.mount('/content/drive')

In [ ]:
with open('image_paths.json', 'r') as f:
    image_paths = json.load(f)

In [ ]:
image_embeddings = np.load('/content/image_embeddings.npy')

# Image Search

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
tokenizer = open_clip.get_tokenizer('ViT-B-32')

In [ ]:
d = image_embeddings.shape[1]
index = faiss.IndexFlatIP(d)
index.add(image_embeddings)

In [ ]:
def search_text(query, top_k=3):
    with torch.no_grad():
        tokenized = tokenizer([query])
        text_embed = model.encode_text(tokenized)
        text_embed = text_embed / text_embed.norm(dim=-1, keepdim=True)
        text_embed_np = text_embed.cpu().numpy()

    D, I = index.search(text_embed_np, top_k)
    return [image_paths[i] for i in I[0]]

In [ ]:
def search_image(query, top_k=3):
  img = PILImage.open(query)
  image_tensor = preprocess(img).unsqueeze(0)

  with torch.no_grad():
    image_embeddings = model.encode_image(image_tensor)

    image_embeddings = image_embeddings / image_embeddings.norm(dim=-1, keepdim=True)  # Normalize
    image_embeddings_np = image_embeddings.cpu().numpy()

  D, I = index.search(image_embeddings_np, top_k)
  return [image_paths[i] for i in I[0]]

In [ ]:
def display_results(matches):
  for path in matches:
    img = PILImage.open(path)
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    display(IPyImage(data=buf.getvalue(), width=400))
    print("\n")

In [ ]:
matched_paths = search_text("cold animal")

display_results(matched_paths)

In [ ]:
matched_paths = search_image("dog.jpg")

display_results(matched_paths)

# Image Captioning

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

processor_cap = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model_cap = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

In [ ]:
def generate_caption(images):
  img = [PILImage.open(img).convert("RGB") for img in images]
  inputs = processor_cap(images=img, return_tensors="pt")
  out = model_cap.generate(**inputs)
  caption = [processor_cap.decode(out[i], skip_special_tokens=True) for i in range(3)]
  return caption


In [ ]:
generate_caption(matched_paths)